🧾 PROJECT PLAN – PUBMED + TRANSFORMER

PHASE 1 — PLANNING
	1.	Define NLP task (e.g., publication type classification)
	2.	Define label set (e.g., Review, Clinical Trial, Case Report, Meta-Analysis, Others)
	3.	Choose API (NCBI Entrez)
	4.	Choose transformer models (e.g., SciBERT + BioBERT + BERT-base)
	5.	Choose metrics (accuracy + F1 + confusion matrix)

⸻

PHASE 2 — DATA COLLECTION
	6.	Query PubMed API for papers
	7.	Download PMIDs for target categories
	8.	Fetch metadata via efetch
	9.	Extract:
	•	title
	•	abstract
	•	publication type
	•	year
	10.	Store raw JSON/XML
	11.	Convert to DataFrame
	12.	Save as raw CSV/JSON

⸻

PHASE 3 — DATA PREPARATION
	13.	Filter papers with missing abstracts
	14.	Filter papers without target labels
	15.	Map labels to numeric IDs
	16.	Remove duplicates
	17.	Remove very short abstracts
	18.	Split into train/val/test sets
	19.	Save cleaned dataset

⸻

PHASE 4 — TEXT PROCESSING
	20.	Tokenize text with transformer tokenizer
	21.	Truncate or pad to max sequence length
	22.	Convert to HuggingFace Dataset format

⸻

PHASE 5 — MODEL TRAINING
	23.	Load transformer model (e.g., SciBERT)
	24.	Configure training arguments
	25.	Fine-tune on train set
	26.	Validate on val set each epoch
	27.	Save best-performing checkpoint

⸻

PHASE 6 — EVALUATION
	28.	Load test set
	29.	Generate predictions
	30.	Compute:

	•	accuracy
	•	precision
	•	recall
	•	F1

	31.	Compute confusion matrix
	32.	Perform error analysis (inspect misclassified samples)

⸻

PHASE 7 — COMPARISON (OPTIONAL BUT STRONG)
	33.	Train baseline transformer (e.g., BERT-base)
	34.	Train domain transformer (e.g., SciBERT/BioBERT)
	35.	Compare performance on test set
	36.	Compare inference time / model size

⸻

PHASE 8 — DOCUMENTATION + BUSINESS VALUE
	37.	Explain medical classification relevance
	38.	Explain independent data collection
	39.	Document API usage
	40.	Document preprocessing choices
	41.	Document transformer architecture briefly
	42.	Document evaluation results
	43.	State limitations
	44.	State possible improvements
	45.	State business/scientific use cases

⸻

PHASE 9 — FINAL ARTIFACTS
	46.	Final cleaned dataset (CSV)
	47.	Training notebook
	48.	Evaluation notebook
	49.	Model result plots
	50.	Report / presentation slides

In [ ]:
#pip install requests pandas sklearn

In [1]:
pip install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SK

In [2]:
import requests
import time
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'sklearn'

## Getting data via API

In [14]:
# CONFIG
EMAIL = "w.madro@student.edu.com"  
RETMAX = 200                   
SAVE_DIR = Path("pubmed_raw")
SAVE_DIR.mkdir(exist_ok=True)

In [15]:
# Search terms for specific publication types
SEARCH_QUERIES = {
    "clinical_trial": "clinical trial[pt]",
    "review": "review[pt]",
    "case_report": "case reports[pt]",
    "letter": "letter[pt]",
    "meta_analysis": "meta-analysis[pt]"
}

In [16]:
def search_pubmed(term, retmax=RETMAX):
    # Return a list of PMIDs for the given PubMed query.
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "retmax": retmax,
        "retmode": "json",
        "email": EMAIL
    }
    r = requests.get(base, params=params)
    r.raise_for_status()
    data = r.json()
    pmids = data["esearchresult"]["idlist"]
    return pmids

In [17]:
def fetch_details(pmids, raw_xml_path=None, sleep_sec=0.4):
    # fetch XML for given PMIDs
    # extract: pmid, title, abstract, pub_types, year
    if not pmids:
        return []

    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "email": EMAIL,
    }

    r = requests.get(base, params=params)
    r.raise_for_status()
    xml_text = r.text

    # optional: save raw XML
    if raw_xml_path is not None:
        raw_xml_path.write_text(xml_text, encoding="utf-8")

    time.sleep(sleep_sec)  # be nice to NCBI

    root = ET.fromstring(xml_text)
    records = []

    for article in root.findall(".//PubmedArticle"):
        pmid   = article.findtext(".//MedlineCitation/PMID")
        title  = article.findtext(".//MedlineCitation/Article/ArticleTitle")

        # abstract: join all AbstractText pieces, if any
        abstract_elems = article.findall(".//MedlineCitation/Article/Abstract/AbstractText")
        abstract = "\n".join(a.text or "" for a in abstract_elems) if abstract_elems else None

        # publication types as list of strings
        pub_types = [pt.text for pt in article.findall(
            ".//MedlineCitation/Article/PublicationTypeList/PublicationType"
        ) if pt is not None and pt.text]

        # year: just take <Year> if present (no MedlineDate fallback)
        year = article.findtext(".//MedlineCitation/Article/Journal/JournalIssue/PubDate/Year")

        records.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "pub_types": pub_types,
            "year": year,
        })

    return records

In [18]:
all_records = []

for label_name, query in SEARCH_QUERIES.items():
    print(f"{label_name}: searching for → {query}")

    pmids = search_pubmed(query, retmax=RETMAX)
    print(f"  PMIDs found: {len(pmids)}")

    # If no PMIDs, fetch_details will just return []
    records = fetch_details(pmids)

    # add label for classification
    for rec in records:
        rec["label_name"] = label_name

    all_records.extend(records)

print(f"Total records collected: {len(all_records)}")

df = pd.DataFrame(all_records)
df = df.dropna(subset=["abstract"]).reset_index(drop=True)

df.to_csv("pubmed_raw_dataset.csv", index=False)
print("Saved: pubmed_raw_dataset.csv")

clinical_trial: searching for → clinical trial[pt]
  PMIDs found: 200
review: searching for → review[pt]
  PMIDs found: 200
case_report: searching for → case reports[pt]
  PMIDs found: 200
letter: searching for → letter[pt]
  PMIDs found: 200
meta_analysis: searching for → meta-analysis[pt]
  PMIDs found: 200
Total records collected: 1000
Saved: pubmed_raw_dataset.csv


## Data inspection

In [22]:
display(df.head(10))

,pmid,title,abstract,pub_types,year,label_name
0,41540080,Safety of Vojta's reflex locomotion in term pr...,The Vojta's method (VRL) is a neurophysiologic...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
1,41538804,Cabergoline for Lactation Inhibition After Ear...,To evaluate cabergoline's efficacy at decreasi...,"[Journal Article, Randomized Controlled Trial,...",2026,clinical_trial
2,41538790,Efficacy of Telehealth-Based Coaching to Impro...,Cancer survivors face significant challenges i...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
3,41538789,Personalized Transdiagnostic Cognitive Behavio...,University students show a high prevalence of ...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
4,41538762,Olaparib in Patients With Solid Tumors With,The Targeted Agent and Profiling Utilization R...,"[Journal Article, Clinical Trial, Phase II]",2026,clinical_trial
5,41538533,Effect of adding indocyanine green to identify...,The aim of the study was to examine the effect...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
6,41538517,Privacy Fact Sheets for Mitigating Disease-Rel...,The German electronic health record (EHR) aims...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
7,41538497,Salivary fluoride bioavailability after applic...,This study aimed to evaluate the bioavailabili...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial
8,41538382,Exploratory Analysis of Urinary and Sexual Dys...,Limited evidence is available on the effects o...,"[Journal Article, Randomized Controlled Trial,...",2026,clinical_trial
9,41538380,Effects of plyometric jump training on measure...,The purpose of this study was to determine the...,"[Journal Article, Randomized Controlled Trial]",2026,clinical_trial


In [ ]:


# 0. Load raw data
df = pd.read_csv("pubmed_raw_dataset.csv")

# 13. Filter papers with missing abstracts
df = df.dropna(subset=["abstract"])

# 14. Filter papers without target labels
df = df.dropna(subset=["label_name"])

# (optional) If you want to keep only specific labels, uncomment and edit:
# target_labels = ["clinical_trial", "review", "case_report", "meta_analysis", "systematic_review"]
# df = df[df["label_name"].isin(target_labels)]

# 15. Map labels to numeric IDs
label_names = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df["label_id"] = df["label_name"].map(label2id)

# 16. Remove duplicates (by abstract)
df = df.drop_duplicates(subset=["abstract"]).reset_index(drop=True)

# 17. Remove very short abstracts (e.g. fewer than 50 tokens)
df["abstract_len"] = df["abstract"].str.split().str.len()
df = df[df["abstract_len"] >= 50].reset_index(drop=True)
df = df.drop(columns=["abstract_len"])

print("Final dataset size:", len(df))
print("Label distribution:")
print(df["label_name"].value_counts())

# 18. Split into train/val/test (e.g. 70/15/15), stratified by label_id
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label_id"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=42,
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

# 19. Save cleaned datasets
train_df.to_csv("pubmed_clean_train.csv", index=False)
val_df.to_csv("pubmed_clean_val.csv", index=False)
test_df.to_csv("pubmed_clean_test.csv", index=False)

# (optional) save label mapping
mapping_df = pd.DataFrame(
    [{"label_id": i, "label_name": name} for name, i in label2id.items()]
).sort_values("label_id")
mapping_df.to_csv("pubmed_label_mapping.csv", index=False)

print("Saved: pubmed_clean_train/val/test.csv and pubmed_label_mapping.csv")